# Separazione delle note

In [173]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from lib.readwav import readwav
import ipywidgets as widgets
import json
import emcee
from scipy.special import logsumexp
from onset_detection import *

In [174]:
%matplotlib widget
plt.close('all')

piano_dir = "dati_piano"
piano_files = [f for f in os.listdir(piano_dir) if f.endswith('.wav')]
piano_files.sort()  # Sort files to ensure correct order

widget_list = []

In [ ]:
file_select = widgets.Dropdown(
    options=piano_files,
    description='Select file:',
    disabled=False,
)
#display(file_select)

In [176]:
fs, waveform = readwav(os.path.join(piano_dir, file_select.value))
waveform = waveform[:,0]  # Use only the first channel if stereo
print(f"Sampling rate: {fs} Hz, Duration: {len(waveform)/fs:.2f} seconds")

Sampling rate: 44100 Hz, Duration: 19.32 seconds


In [177]:
frame_size_select = widgets.IntSlider(
    value=2048,
    min=512,
    max=8192,
    step=512,
    description='Frame Size:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)

hop_size_select = widgets.IntSlider(
    value=512,
    min=128,
    max=2048,
    step=128,
    description='Hop Size:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)
#display(frame_size_select, hop_size_select)
widget_list.extend([file_select, frame_size_select, hop_size_select])

In [178]:
spectral_flux = compute_spectral_flux(waveform, frame_size=frame_size_select.value, hop_size=hop_size_select.value)

In [ ]:
convolve_window_select = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Convolve Window:',
    disabled=False,    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)
prominence_select = widgets.IntSlider(
    value=1000,
    min=100,
    max=5000,
    step=100,
    description='Prominence:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)
distance_select = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description='Distance:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
)
#display(convolve_window_select, prominence_select, distance_select)
widget_list.extend([convolve_window_select, prominence_select, distance_select])

In [180]:
onsets = detect_onsets(waveform, fs,
                       convolve_window=convolve_window_select.value, prominence=prominence_select.value,
                       distance=distance_select.value, frame_size=frame_size_select.value,
                       hop_size=hop_size_select.value)


In [181]:
delta_select = widgets.FloatSlider(
    value=0.1,
    min=0.01,
    max=0.5,
    step=0.01,
    description='Delta (s):',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
#display(delta_select)
widget_list.append(delta_select)

In [182]:
segmented_waveform = segment_signal(waveform, fs, onsets, delta=0.1)

In [183]:
with plt.ioff():

    fig, axes = plt.subplots(2, 1, figsize=(10, 6))
    canvas = fig.canvas

    axes[0].set_title(f"Segmented waveform for {file_select.value}")

    axes[1].set_title("Spectral Flux with Detected Onsets")

output_widget = widgets.Output()

def update_plots(*args):

    try:

        fs, waveform = readwav(os.path.join(piano_dir, file_select.value))
        waveform = waveform[:,0]  # Use only the first channel if stereo

        inds, spectral_flux = compute_spectral_flux(waveform, frame_size=frame_size_select.value, hop_size=hop_size_select.value)
        smoothed_flux = sp.ndimage.uniform_filter1d(spectral_flux, size=convolve_window_select.value)

        onsets = detect_onsets(waveform, fs,
                            convolve_window=convolve_window_select.value, prominence=prominence_select.value,
                            distance=distance_select.value, frame_size=frame_size_select.value,
                            hop_size=hop_size_select.value)

        onsets = np.array(onsets)  # Ensure onsets is a numpy array for indexing

        segmented_waveform = segment_signal(waveform, fs, onsets, delta=delta_select.value)

        times = samples_to_secs(np.arange(len(waveform)), fs)
        spectral_flux_times = samples_to_secs(inds, fs)

        for ax in axes:
            ax.clear()

        axes[0].set_title(f"Segmented waveform for {file_select.value}")
        axes[0].plot(times, waveform, label='Waveform')

        # draw segments
        for i, (start, end) in enumerate(zip(onsets[:-1], onsets[1:])):
            axes[0].axvspan(times[start] + delta_select.value, times[end] - delta_select.value, alpha=0.5, color=f'C{i}')
            axes[0].text((times[start] + times[end]) / 2 - delta_select.value, np.max(waveform)*0.8, f'{i+1}', color='black')

        axes[1].set_title("Spectral Flux with Detected Onsets")
        #axes[1].plot(spectral_flux_times, spectral_flux, label='Spectral Flux')
        axes[1].plot(spectral_flux_times, smoothed_flux, label='Smoothed Spectral Flux')

        canvas.draw_idle()

        with output_widget:
            output_widget.clear_output()
            print(f"Updated plots for {file_select.value}")
            print(f"Detected {len(onsets)} onsets at:")
            segment_lenghts = np.diff(onsets) / fs
            [print(f"  - {times[onset]:.2f} seconds, Length: {segment_lenghts[i]:.2f} seconds") for i, onset in enumerate(onsets[:-1])]
    
    except Exception as e:
        with output_widget:
            print(f"Error updating plots: {e}")


for widget in widget_list:
    widget.observe(update_plots, names='value')




In [184]:
widget_layout = widgets.VBox(widget_list + [output_widget])
all_layout = widgets.HBox([widget_layout, fig.canvas])
display(all_layout)
update_plots()